# 04 - Multi-factor combination & factor timing

Three combination recipes, side by side:
1. **Equal-weight** -- the no-prior baseline.
2. **Static IC weighting** -- weight each factor by its full-sample mean IC (look-ahead!).
3. **Dynamic rolling-Sharpe timing** -- per-date weights from the past 63d L-S Sharpe.

We measure each composite signal's IC and the resulting long-short PnL.

In [ ]:
import sys, os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings; warnings.filterwarnings("ignore")

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))

from mfa.config import load_config
from mfa.viz import apply_style
apply_style()

cfg = load_config(ROOT / "configs" / "default.yaml")
print("project:", cfg.project_root.name, "| processed:", cfg.processed_dir.exists())


In [ ]:
from mfa.data import build_panels

def _load_raw_panels():
    out = {}
    for k in ["open","high","low","close","volume","vwap","dollar_volume"]:
        p = cfg.processed_dir / f"panel_{k}.parquet"
        if p.exists():
            df = pd.read_parquet(p)
            df.index = pd.to_datetime(df.index, utc=True)
            out[k] = df
    if not out:
        raise RuntimeError("No cached panels found. Run scripts/01_download_data.py first.")
    return out

def _load_factors():
    d = cfg.processed_dir / "factors"
    out = {}
    for p in sorted(d.glob("*.parquet")):
        df = pd.read_parquet(p)
        df.index = pd.to_datetime(df.index, utc=True)
        out[p.stem] = df
    return out

raw_panels = _load_raw_panels()
benchmark = cfg.section("universe").get("benchmark", "SPY")
panels = build_panels(raw_panels, benchmark=benchmark, min_history_days=252)
factors = _load_factors()
print(f"loaded {len(panels)} panels and {len(factors)} factors")
print("date range:", panels["close"].index.min().date(), "->", panels["close"].index.max().date())
print("universe size:", panels["close"].shape[1], " benchmark:", benchmark)


In [ ]:
from mfa.data import forward_returns
from mfa.evaluation import information_coefficient, ic_summary, quantile_returns, long_short_returns
from mfa.evaluation.stats import perf_stats
from mfa.portfolio import combine_factors, factor_timing_weights

H = 5
fwd = {h: forward_returns(panels["close"], [h])[h] for h in [1, H]}

# Per-factor L-S PnL needed for IR-based timing
fpnl = pd.DataFrame({n: long_short_returns(quantile_returns(f, fwd[1], 5)) for n, f in factors.items()})
fic  = pd.DataFrame({n: information_coefficient(f, fwd[1]) for n, f in factors.items()})
print("factor PnL panel shape:", fpnl.shape)


## 1. Three composites

In [ ]:
composites = {}

# (a) equal weight
composites["equal"] = combine_factors(factors, weights=None, standardize=True)

# (b) static IC weighting (in-sample, look-ahead biased -- diagnostic only)
ic_means = {n: information_coefficient(factors[n], fwd[H]).mean() for n in factors}
total = sum(abs(v) for v in ic_means.values())
ic_w = {n: v / total for n, v in ic_means.items()}
composites["static_ic"] = combine_factors(factors, weights=ic_w, standardize=True)

# (c) dynamic rolling Sharpe timing
fw = factor_timing_weights(fpnl, factor_ic=fic, method="rolling_sharpe", lookback=63)
composites["timing_sharpe63"] = combine_factors(factors, weights=fw, standardize=True)

for k, v in composites.items():
    print(f"{k:18s}  shape={v.shape}  late nonNaN row count = {v.notna().sum(axis=1).iloc[-1]}")


## 2. Composite IC comparison

In [ ]:
rows = []
for k, c in composites.items():
    ic = information_coefficient(c, fwd[H])
    qr = quantile_returns(c, fwd[H], 5)
    ls = long_short_returns(qr)
    perf = perf_stats(ls, periods_per_year=252/H)
    rows.append({"composite": k, "ic_mean": ic.mean(), "ic_ir": ic.mean()/ic.std(),
                 "ic_t": (ic.mean()/ic.std())*np.sqrt(ic.notna().sum()),
                 "ls_sharpe": perf["sharpe"], "ls_ann": perf["ann_return"], "ls_mdd": perf["max_drawdown"]})
df = pd.DataFrame(rows).set_index("composite")
df.style.format({"ic_mean":"{:+.4f}","ic_ir":"{:+.2f}","ic_t":"{:+.2f}",
                 "ls_sharpe":"{:+.2f}","ls_ann":"{:+.2%}","ls_mdd":"{:.1%}"}).background_gradient(cmap="RdYlGn", subset=["ls_sharpe","ic_t"])


## 3. L-S equity comparison

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4.5))
colors = {"equal":"#bbbbbb","static_ic":"#E63946","timing_sharpe63":"#2E86AB"}
for k, c in composites.items():
    qr = quantile_returns(c, fwd[H], 5)
    ls = long_short_returns(qr)
    (1 + ls.fillna(0)).cumprod().plot(ax=ax, label=k, color=colors[k], lw=1.8)
ax.set_title("Composite long-short equity"); ax.set_ylabel("growth of $1"); ax.legend(); plt.show()


## 4. Visualising the dynamic factor weights

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4.5))
fw.plot.area(ax=ax, lw=0, alpha=0.85, colormap="tab20")
ax.set_title("Per-factor weight (rolling-Sharpe timing, 63d)")
ax.set_ylabel("weight"); ax.set_ylim(0, 1)
ax.legend(loc="center left", bbox_to_anchor=(1.0, 0.5), ncol=1, fontsize=7); plt.tight_layout(); plt.show()


## 5. Time-varying tilt: what the model thinks today

In [ ]:
last_w = fw.iloc[-1].sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(8, max(4, 0.25*len(last_w))))
last_w.plot.barh(ax=ax, color="#2E86AB", edgecolor="black")
ax.set_title(f"Factor weights on {fw.index[-1].date()}"); ax.set_xlabel("weight"); plt.show()


### Takeaways

* The static IC weighting is *upper-bound diagnostic*; it uses full-sample knowledge.
* Dynamic rolling-Sharpe timing concentrates weight on whichever factor has been working
  recently. This adds turnover but adapts to regime changes.
* Equal-weighting is the conservative baseline; never wins, never blows up.